In [40]:
from IPython.display import display, Math, Latex

import pandas as pd
import numpy as np
import numpy_financial as npf
import yfinance as yf
import matplotlib.pyplot as plt
import random
from datetime import datetime

Read the tickers from a file

In [41]:
# Read the supplied ticker file. It may have a header or just one column of ticker symbols.
ticker_data = pd.read_csv("Tickers.csv", header=None)
tickers = ticker_data.iloc[:, 0].dropna().astype(str).str.strip()

if not tickers.empty and tickers.iloc[0].lower() in ["ticker", "symbol"]:
    tickers = tickers.iloc[1:]

tickers = tickers[tickers != ""].drop_duplicates()

raw_info = {}
for ticker in tickers:
    try:
        raw_info[ticker] = yf.Ticker(ticker).info
    except Exception:
        raw_info[ticker] = {}

In [42]:
type_s= []
currency = []
market = []
sector = []

for ticker in tickers: 
    try: 
        tick = raw_info.get(ticker)
        type_s.append(tick.get('quoteType'))
        currency.append(tick.get('currency'))
        market.append(tick.get('market'))
        sector.append(tick.get('sector'))
    except:
        type_s.append(None)
        currency.append(None)
        market.append(None)
        sector.append(None)

info = {
    'Type' : type_s,
    'Currency' : currency,
    'Market' : market,
    'Sector' : sector
}

all_stocks = pd.DataFrame(info, index=tickers)
all_stocks.index.name = 'Ticker'
all_stocks

,Type,Currency,Market,Sector
Ticker,,,,
AAPL,EQUITY,USD,us_market,Technology
ABBV,EQUITY,USD,us_market,Healthcare
ABT,EQUITY,USD,us_market,Healthcare
ACN,EQUITY,USD,us_market,Technology
AGN,NONE,None,None,None
AIG,EQUITY,USD,us_market,Financial Services
AMZN,EQUITY,USD,us_market,Consumer Cyclical
AXP,EQUITY,USD,us_market,Financial Services
BA,EQUITY,USD,us_market,Industrials


check if it is eligible

In [43]:
mask = all_stocks['Type'] == 'EQUITY'
mask1 = all_stocks['Currency'].isin(['USD', 'CAD'])
mask2 = all_stocks['Market'].isin(['us_market', 'ca_market'])

eligible_stocks = all_stocks[mask & mask1 & mask2]
eligible_stocks

,Type,Currency,Market,Sector
Ticker,,,,
AAPL,EQUITY,USD,us_market,Technology
ABBV,EQUITY,USD,us_market,Healthcare
ABT,EQUITY,USD,us_market,Healthcare
ACN,EQUITY,USD,us_market,Technology
AIG,EQUITY,USD,us_market,Financial Services
AMZN,EQUITY,USD,us_market,Consumer Cyclical
AXP,EQUITY,USD,us_market,Financial Services
BA,EQUITY,USD,us_market,Industrials
BAC,EQUITY,USD,us_market,Financial Services


check for volume

In [44]:
# Get data from the beginning of the volume window through the purchase date.
buy_date = "2026-11-27"
global_start = "2025-10-01"
global_end = (pd.Timestamp(buy_date) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")

tickers_list = eligible_stocks.index.tolist().copy()
hist_data = yf.download(
    tickers_list,
    start=global_start,
    end=global_end,
    group_by="ticker"
)

[*********************100%***********************]  35 of 35 completed


In [59]:
start='2025-10-01'
end='2026-10-01'
min_avg_volume=5000

for ticker in eligible_stocks.index.tolist():
    history = hist_data[ticker].loc[start:end]
    history['Month'] = history.index.strftime('%Y-%m')
    
    month_counts = history.groupby('Month').size()
    valid_months = month_counts[month_counts >= 18].index
    history = history[history['Month'].isin(valid_months)]

    eligible_stocks.loc[ticker, 'Avg_Volume'] = history.Volume.mean()

mask = eligible_stocks['Avg_Volume']>=min_avg_volume
eligible_stocks = eligible_stocks[mask]

eligible_stocks

,Type,Currency,Market,Sector,Avg_Volume,Close,Close_CAD,Beta,Return %,Score
Ticker,,,,,,,,,,
AAPL,EQUITY,USD,us_market,Technology,4.828203e+07,346.836487,486.008102,0.622044,29.601635,0.469210
ABBV,EQUITY,USD,us_market,Healthcare,6.281745e+06,286.390015,401.306877,-0.088480,29.351972,0.331214
ABT,EQUITY,USD,us_market,Healthcare,1.008423e+07,103.099998,144.469906,0.008458,-16.918916,0.181048
ACN,EQUITY,USD,us_market,Technology,6.116120e+06,235.160004,329.520311,0.108679,-3.524663,0.249267
AIG,EQUITY,USD,us_market,Financial Services,4.324195e+06,79.580002,111.512275,0.016578,2.812180,0.254624
AMZN,EQUITY,USD,us_market,Consumer Cyclical,4.652561e+07,253.100006,354.658919,1.455028,11.512537,0.563905
AXP,EQUITY,USD,us_market,Financial Services,3.054224e+06,312.910004,438.468277,0.923310,-12.507702,0.373654
BA,EQUITY,USD,us_market,Industrials,7.140954e+06,168.179993,235.663900,1.209087,-24.582963,0.384721
BAC,EQUITY,USD,us_market,Financial Services,3.673368e+07,50.680000,71.015858,0.724003,-2.375913,0.372177


take the closing price

In [46]:
end_date = (pd.Timestamp(buy_date) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")

for ticker in eligible_stocks.index.tolist():
    history = hist_data[ticker].loc[buy_date:end_date]
    if not history.empty:
        eligible_stocks.loc[ticker, "Close"] = history.loc[buy_date, "Close"]

eligible_stocks

,Type,Currency,Market,Sector,Avg_Volume,Close
Ticker,,,,,,
AAPL,EQUITY,USD,us_market,Technology,4.828203e+07,346.836487
ABBV,EQUITY,USD,us_market,Healthcare,6.281745e+06,286.390015
ABT,EQUITY,USD,us_market,Healthcare,1.008423e+07,103.099998
ACN,EQUITY,USD,us_market,Technology,6.116120e+06,235.160004
AIG,EQUITY,USD,us_market,Financial Services,4.324195e+06,79.580002
AMZN,EQUITY,USD,us_market,Consumer Cyclical,4.652561e+07,253.100006
AXP,EQUITY,USD,us_market,Financial Services,3.054224e+06,312.910004
BA,EQUITY,USD,us_market,Industrials,7.140954e+06,168.179993
BAC,EQUITY,USD,us_market,Financial Services,3.673368e+07,50.680000


convert to CAD

In [47]:
usdcad = yf.Ticker('USDCAD=X').history(start=buy_date, end=end_date)
rate = usdcad.loc[buy_date, 'Close']
for ticker in eligible_stocks.index.tolist():
    price = eligible_stocks.loc[ticker, 'Close']
    if eligible_stocks.loc[ticker, 'Currency'] == 'USD':
        eligible_stocks.loc[ticker, 'Close_CAD'] = price * rate
    else :
        eligible_stocks.loc[ticker, 'Close_CAD'] = price

eligible_stocks

,Type,Currency,Market,Sector,Avg_Volume,Close,Close_CAD
Ticker,,,,,,,
AAPL,EQUITY,USD,us_market,Technology,4.828203e+07,346.836487,486.008102
ABBV,EQUITY,USD,us_market,Healthcare,6.281745e+06,286.390015,401.306877
ABT,EQUITY,USD,us_market,Healthcare,1.008423e+07,103.099998,144.469906
ACN,EQUITY,USD,us_market,Technology,6.116120e+06,235.160004,329.520311
AIG,EQUITY,USD,us_market,Financial Services,4.324195e+06,79.580002,111.512275
AMZN,EQUITY,USD,us_market,Consumer Cyclical,4.652561e+07,253.100006,354.658919
AXP,EQUITY,USD,us_market,Financial Services,3.054224e+06,312.910004,438.468277
BA,EQUITY,USD,us_market,Industrials,7.140954e+06,168.179993,235.663900
BAC,EQUITY,USD,us_market,Financial Services,3.673368e+07,50.680000,71.015858


fee application function

PORTFOLIO A

calculate Beta using top 5000 stocks

In [49]:
# Estimate beta manually using the year before the purchase date and the S&P 500 as the market benchmark.
beta_start = (pd.Timestamp(buy_date) - pd.DateOffset(years=1)).strftime("%Y-%m-%d")
market_hist = yf.download("^GSPC", start=beta_start, end=end_date)
market_close = market_hist["Close"]["^GSPC"]
market_returns = market_close.pct_change()

for ticker in eligible_stocks.index.tolist():
    stock_close = hist_data[ticker]["Close"].loc[beta_start:buy_date]
    stock_returns = stock_close.pct_change(fill_method=None)

    return_data = pd.concat(
        [stock_returns, market_returns],
        axis=1,
        join="inner"
    ).dropna()

    covariance = return_data.iloc[:, 0].cov(return_data.iloc[:, 1])
    variance = return_data.iloc[:, 1].var()

    if variance != 0:
        eligible_stocks.loc[ticker, "Beta"] = covariance / variance

eligible_stocks

[*********************100%***********************]  1 of 1 completed


,Type,Currency,Market,Sector,Avg_Volume,Close,Close_CAD,Beta
Ticker,,,,,,,,
AAPL,EQUITY,USD,us_market,Technology,4.828203e+07,346.836487,486.008102,0.622044
ABBV,EQUITY,USD,us_market,Healthcare,6.281745e+06,286.390015,401.306877,-0.088480
ABT,EQUITY,USD,us_market,Healthcare,1.008423e+07,103.099998,144.469906,0.008458
ACN,EQUITY,USD,us_market,Technology,6.116120e+06,235.160004,329.520311,0.108679
AIG,EQUITY,USD,us_market,Financial Services,4.324195e+06,79.580002,111.512275,0.016578
AMZN,EQUITY,USD,us_market,Consumer Cyclical,4.652561e+07,253.100006,354.658919,1.455028
AXP,EQUITY,USD,us_market,Financial Services,3.054224e+06,312.910004,438.468277,0.923310
BA,EQUITY,USD,us_market,Industrials,7.140954e+06,168.179993,235.663900,1.209087
BAC,EQUITY,USD,us_market,Financial Services,3.673368e+07,50.680000,71.015858,0.724003


If we couldn't calculate a beta ourselves, use Yahoo's beta instead. If Yahoo doesn't have one either, just assume average risk (1.0)

In [50]:
for ticker in eligible_stocks.index.tolist():
    beta = eligible_stocks.loc[ticker, 'Beta']

    if pd.isna(beta):
        yahoo_beta = raw_info[ticker].get('beta')
        if yahoo_beta is None:
            beta = 1.0
        else:
            beta = yahoo_beta
        eligible_stocks.loc[ticker, 'Beta'] = beta

eligible_stocks

,Type,Currency,Market,Sector,Avg_Volume,Close,Close_CAD,Beta
Ticker,,,,,,,,
AAPL,EQUITY,USD,us_market,Technology,4.828203e+07,346.836487,486.008102,0.622044
ABBV,EQUITY,USD,us_market,Healthcare,6.281745e+06,286.390015,401.306877,-0.088480
ABT,EQUITY,USD,us_market,Healthcare,1.008423e+07,103.099998,144.469906,0.008458
ACN,EQUITY,USD,us_market,Technology,6.116120e+06,235.160004,329.520311,0.108679
AIG,EQUITY,USD,us_market,Financial Services,4.324195e+06,79.580002,111.512275,0.016578
AMZN,EQUITY,USD,us_market,Consumer Cyclical,4.652561e+07,253.100006,354.658919,1.455028
AXP,EQUITY,USD,us_market,Financial Services,3.054224e+06,312.910004,438.468277,0.923310
BA,EQUITY,USD,us_market,Industrials,7.140954e+06,168.179993,235.663900,1.209087
BAC,EQUITY,USD,us_market,Financial Services,3.673368e+07,50.680000,71.015858,0.724003


1 year returns

In [54]:
one_year_ago = (pd.Timestamp(buy_date) - pd.DateOffset(years=1)).strftime('%Y-%m-%d')

for ticker in eligible_stocks.index.tolist():
    history = hist_data[ticker]
    if one_year_ago in history.index and buy_date in history.index:
        old_price = history.loc[one_year_ago, 'Close']
        new_price = history.loc[buy_date, 'Close']
        eligible_stocks.loc[ticker, 'Return %'] = (new_price - old_price) / old_price * 100
    else:
        old_price = history['Close'].dropna().asof(pd.Timestamp(one_year_ago))
        new_price = history['Close'].dropna().asof(pd.Timestamp(buy_date))

    if pd.notna(old_price) and pd.notna(new_price):
        eligible_stocks.loc[ticker, 'Return %'] = (new_price - old_price) / old_price * 100
    else:
        eligible_stocks.loc[ticker, 'Return %'] = None

# Drop any stock we couldn't compute a 1-year return for (missing data that far back)
eligible_stocks = eligible_stocks.dropna(subset=['Return %'])
eligible_stocks

,Type,Currency,Market,Sector,Avg_Volume,Close,Close_CAD,Beta,Return %
Ticker,,,,,,,,,
AAPL,EQUITY,USD,us_market,Technology,4.828203e+07,346.836487,486.008102,0.622044,29.601635
ABBV,EQUITY,USD,us_market,Healthcare,6.281745e+06,286.390015,401.306877,-0.088480,29.351972
ABT,EQUITY,USD,us_market,Healthcare,1.008423e+07,103.099998,144.469906,0.008458,-16.918916
ACN,EQUITY,USD,us_market,Technology,6.116120e+06,235.160004,329.520311,0.108679,-3.524663
AIG,EQUITY,USD,us_market,Financial Services,4.324195e+06,79.580002,111.512275,0.016578,2.812180
AMZN,EQUITY,USD,us_market,Consumer Cyclical,4.652561e+07,253.100006,354.658919,1.455028,11.512537
AXP,EQUITY,USD,us_market,Financial Services,3.054224e+06,312.910004,438.468277,0.923310,-12.507702
BA,EQUITY,USD,us_market,Industrials,7.140954e+06,168.179993,235.663900,1.209087,-24.582963
BAC,EQUITY,USD,us_market,Financial Services,3.673368e+07,50.680000,71.015858,0.724003,-2.375913


Combine Beta and Return into one Score

In [55]:
beta_norm = (eligible_stocks['Beta'] - eligible_stocks['Beta'].min()) / (eligible_stocks['Beta'].max() - eligible_stocks['Beta'].min())
return_norm = (eligible_stocks['Return %'] - eligible_stocks['Return %'].min()) / (eligible_stocks['Return %'].max() - eligible_stocks['Return %'].min())

eligible_stocks['Score'] = (beta_norm + return_norm) / 2
eligible_stocks.sort_values('Score', ascending=False)

,Type,Currency,Market,Sector,Avg_Volume,Close,Close_CAD,Beta,Return %,Score
Ticker,,,,,,,,,,
BB.TO,EQUITY,CAD,ca_market,Technology,3.406802e+06,13.390000,13.390000,1.719316,99.850757,0.937293
TXN,EQUITY,USD,us_market,Technology,7.674424e+06,307.409576,430.760749,1.444344,87.066793,0.837585
CAT,EQUITY,USD,us_market,Industrials,2.807785e+06,713.309998,999.532780,1.747104,36.485816,0.711399
QCOM,EQUITY,USD,us_market,Technology,1.412401e+07,184.169998,258.070055,2.044333,0.173327,0.636220
TD.TO,EQUITY,CAD,ca_market,Financial Services,5.614886e+06,176.570007,176.570007,0.682420,59.194990,0.588862
SHOP.TO,EQUITY,CAD,ca_market,Technology,1.943639e+06,232.929993,232.929993,1.864963,-4.992456,0.582760
C,EQUITY,USD,us_market,Financial Services,1.269748e+07,122.019730,170.981369,1.276123,24.057317,0.575171
AMZN,EQUITY,USD,us_market,Consumer Cyclical,4.652561e+07,253.100006,354.658919,1.455028,11.512537,0.563905
BIIB,EQUITY,USD,us_market,Healthcare,1.408238e+06,252.729996,354.140438,0.348119,68.005058,0.556517


Split into a growth bucket (Technology + Communication Services, target 40%) and everything else (target 60%). If there aren't enough growth stocks, use fewer and lower the target so we never break the 15% per-stock cap

In [56]:
#select sectors the portfolio is going to focus on
growth_sectors = ['Technology', 'Communication Services']

mask = eligible_stocks['Sector'].isin(growth_sectors)
growth_stocks = eligible_stocks[mask].sort_values('Score', ascending=False)
other_stocks = eligible_stocks[~mask].sort_values('Score', ascending=False)

# total stock count
n_total = 15
# use 5 selected sector stocks, or fewer if the ticker file doesn't have 5
n_growth = min(5, len(growth_stocks))   
n_other = n_total - n_growth

#take the n_growth number of the best stocks in the selected sectors
selected_growth = growth_stocks.head(n_growth).copy()
selected_other = other_stocks.head(n_other).copy()

# Quick sanity check: make sure no OTHER single sector is secretly dominating this bucket
selected_other['Sector'].value_counts()

Sector
Financial Services    4
Healthcare            3
Industrials           2
Consumer Cyclical     1
Name: count, dtype: int64

Weight each bucket by Score, then cap any single stock at 15% and hand the extra weight to the rest of its bucket

In [57]:
def cap_weights(df, cap=0.15):
    # add 0.0001 to prevent infinite loops due to rounding
    while (df['Weight'] > cap + 0.0001).any():
        excess = 0
        for ticker in df.index:
            if df.loc[ticker, 'Weight'] > cap:
                excess = excess + (df.loc[ticker, 'Weight'] - cap)
                df.loc[ticker, 'Weight'] = cap
    
        not_capped = df[df['Weight'] < cap]
        for ticker in not_capped.index:
            df.loc[ticker, 'Weight'] = df.loc[ticker, 'Weight'] + excess / len(not_capped)

    return df
    
# can't ask for more than the 15% cap allows with this many stocks
target_growth = min(0.40, n_growth * 0.15)
target_other = 1 - target_growth

selected_growth['Weight'] = selected_growth['Score'] / selected_growth['Score'].sum() * target_growth
selected_other['Weight'] = selected_other['Score'] / selected_other['Score'].sum() * target_other

selected_growth = cap_weights(selected_growth)
selected_other = cap_weights(selected_other)

portfolio_A = pd.concat([selected_growth, selected_other])

print('Total weight:', portfolio_A['Weight'].sum())
print('Growth sector weight:', selected_growth['Weight'].sum())
portfolio_A[['Sector', 'Beta', 'Return %', 'Score', 'Weight']].sort_values('Weight', ascending=False)

Total weight: 1.0000000000000002
Growth sector weight: 0.39999999999999997


,Sector,Beta,Return %,Score,Weight
Ticker,,,,,
BB.TO,Technology,1.719316,99.850757,0.937293,0.108262
TXN,Technology,1.444344,87.066793,0.837585,0.096745
CAT,Industrials,1.747104,36.485816,0.711399,0.077654
QCOM,Technology,2.044333,0.173327,0.636220,0.073486
SHOP.TO,Technology,1.864963,-4.992456,0.582760,0.067311
TD.TO,Financial Services,0.682420,59.194990,0.588862,0.064278
C,Financial Services,1.276123,24.057317,0.575171,0.062784
AMZN,Consumer Cyclical,1.455028,11.512537,0.563905,0.061554
BIIB,Healthcare,0.348119,68.005058,0.556517,0.060748


In [ ]:
fee_rate = 0.001 
def calc_fee_cad(shares_num):
    return min(shares_num * fee_rate, 2.15) * rate

In [61]:
budget = 1000000
def buy_stocks(portfolio, eligible):
    # first pass: split the budget by weight, ignoring fees
    for ticker in portfolio.index.tolist():
        money_for_stock = portfolio.loc[ticker, 'Weight'] * budget
        stock_n = money_for_stock / eligible.loc[ticker, 'Close_CAD']
        portfolio.loc[ticker, 'Shares'] = stock_n

    # add up what the fees would come out to with these share counts
    total_fees = 0
    for ticker in portfolio.index.tolist():
        fee = calc_fee_cad(portfolio.loc[ticker, 'Shares'])
        total_fees = total_fees + fee

    # second pass: shrink the budget by the fees, then size shares again
    investable = budget - total_fees
    for ticker in portfolio.index.tolist():
        money_for_stock = portfolio.loc[ticker, 'Weight'] * investable
        stock_n = money_for_stock / eligible.loc[ticker, 'Close_CAD']
        portfolio.loc[ticker, 'Shares'] = stock_n

    # final value + fee per stock, and a check that we spent close to budget
    total_value = 0
    total_fees_final = 0
    for ticker in portfolio.index.tolist():
        value = portfolio.loc[ticker, 'Shares'] * eligible.loc[ticker, 'Close_CAD']
        fee = calc_fee_cad(portfolio.loc[ticker, 'Shares'])
        portfolio.loc[ticker, 'Value'] = value
        portfolio.loc[ticker, 'Fee'] = fee
        total_value = total_value + value
        total_fees_final = total_fees_final + fee

    print('Total spent:', total_value + total_fees_final)
    print('Budget:', budget)
    print('Leftover:', budget - total_value - total_fees_final)

    return portfolio

portfolio_A = buy_stocks(portfolio_A, eligible_stocks)
portfolio_A

Total spent: 999999.9999599984
Budget: 1000000
Leftover: 4.000160805972541e-05


,Type,Currency,Market,Sector,Avg_Volume,Close,Close_CAD,Beta,Return %,Score,Weight,Shares,Value,Fee
Ticker,,,,,,,,,,,,,,
BB.TO,EQUITY,CAD,ca_market,Technology,3.406802e+06,13.390000,13.390000,1.719316,99.850757,0.937293,0.108262,8085.190580,108260.704645,3.012709
TXN,EQUITY,USD,us_market,Technology,7.674424e+06,307.409576,430.760749,1.444344,87.066793,0.837585,0.096745,224.588867,96744.068372,0.314707
QCOM,EQUITY,USD,us_market,Technology,1.412401e+07,184.169998,258.070055,2.044333,0.173327,0.636220,0.073486,284.750840,73485.664953,0.399010
SHOP.TO,EQUITY,CAD,ca_market,Technology,1.943639e+06,232.929993,232.929993,1.864963,-4.992456,0.582760,0.067311,288.974631,67310.858742,0.404929
AAPL,EQUITY,USD,us_market,Technology,4.828203e+07,346.836487,486.008102,0.622044,29.601635,0.469210,0.054196,111.511516,54195.500106,0.156257
CAT,EQUITY,USD,us_market,Industrials,2.807785e+06,713.309998,999.532780,1.747104,36.485816,0.711399,0.077654,77.689706,77653.407467,0.108863
TD.TO,EQUITY,CAD,ca_market,Financial Services,5.614886e+06,176.570007,176.570007,0.682420,59.194990,0.588862,0.064278,364.035949,64277.830187,0.510109
C,EQUITY,USD,us_market,Financial Services,1.269748e+07,122.019730,170.981369,1.276123,24.057317,0.575171,0.062784,367.194084,62783.346946,0.514534
AMZN,EQUITY,USD,us_market,Consumer Cyclical,4.652561e+07,253.100006,354.658919,1.455028,11.512537,0.563905,0.061554,173.557135,61553.585928,0.243199


In [74]:
Portfolio_Final_A_data = {
    'Ticker' : portfolio_A.index,
    'Price' : portfolio_A.Close.values,
    'Currency' : portfolio_A.Currency.values,
    'Shares' : portfolio_A.Shares.values,
    'Value' : portfolio_A.Value.values,
    # to reduce fees from the total value of the invested shares
    'Weight' : (portfolio_A.Value / portfolio_A.Value.sum()).values
}

Portfolio_Final_A = pd.DataFrame(Portfolio_Final_A_data, index = range(1, len(portfolio_A) + 1))
Portfolio_Final_A

,Ticker,Price,Currency,Shares,Value,Weight
1,BB.TO,13.390000,CAD,8085.190580,108260.704645,0.108262
2,TXN,307.409576,USD,224.588867,96744.068372,0.096745
3,QCOM,184.169998,USD,284.750840,73485.664953,0.073486
4,SHOP.TO,232.929993,CAD,288.974631,67310.858742,0.067311
5,AAPL,346.836487,USD,111.511516,54195.500106,0.054196
6,CAT,713.309998,USD,77.689706,77653.407467,0.077654
7,TD.TO,176.570007,CAD,364.035949,64277.830187,0.064278
8,C,122.019730,USD,367.194084,62783.346946,0.062784
9,AMZN,253.100006,USD,173.557135,61553.585928,0.061554
10,BIIB,252.729996,USD,171.534001,60747.126273,0.060748


In [ ]:
# Compare the current weights with an equal-weight version of the same 15 stocks.

fx_prices = yf.Ticker("USDCAD=X").history(
    start=one_year_ago,
    end=(pd.Timestamp(buy_date) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
)["Close"]

fx_prices.index = pd.to_datetime(fx_prices.index).tz_localize(None)
price_history_cad = pd.DataFrame()

for ticker in portfolio_A.index:
    stock_prices = hist_data[ticker]["Close"].loc[one_year_ago:buy_date].copy()
    stock_prices.index = pd.to_datetime(stock_prices.index).tz_localize(None)

    if portfolio_A.loc[ticker, "Currency"] == "USD":
        fx_for_stock = fx_prices.reindex(stock_prices.index).ffill()
        stock_prices = stock_prices * fx_for_stock

    price_history_cad[ticker] = stock_prices

price_history_cad = price_history_cad.dropna()
growth_of_1_dollar = price_history_cad / price_history_cad.iloc[0]

score_weighted_value = pd.Series(0.0, index=growth_of_1_dollar.index)
equal_weight_value = pd.Series(0.0, index=growth_of_1_dollar.index)

for ticker in portfolio_A.index:
    score_weighted_value = score_weighted_value + (
        growth_of_1_dollar[ticker] * portfolio_A.loc[ticker, "Weight"]
    )
    equal_weight_value = equal_weight_value + (
        growth_of_1_dollar[ticker] / len(portfolio_A)
    )

score_weighted_returns = score_weighted_value.pct_change().dropna()
equal_weight_returns = equal_weight_value.pct_change().dropna()

comparison_A = pd.DataFrame(
    {
        "Historical price return": [
            score_weighted_value.iloc[-1] - 1,
            equal_weight_value.iloc[-1] - 1,
        ],
        "Daily CAD volatility": [
            score_weighted_returns.std(),
            equal_weight_returns.std(),
        ],
    },
    index=["Portfolio A weights", "Same stocks, equal weight"],
)

display(comparison_A * 100)
print("Comparison period:", price_history_cad.index[0].date(), "to", price_history_cad.index[-1].date())


# Final Explanations — Portfolio A

### Explanation 1 — Why Portfolio A makes financial sense

We designed Portfolio A for a client who wants growth and is comfortable taking on more risk. Our strategy gives each eligible stock a score based on its previous-year return and beta. We scale both measures to the same range, then average them so neither has more influence just because it uses a different scale.

We estimate beta manually using about one year of the stock’s daily returns and the S&P 500’s daily returns. Beta measures how sensitive a stock has been to market movements. A higher beta means the stock has tended to move more than the market, which can mean larger gains when the market rises and larger losses when it falls. Investors generally expect more return for accepting more market risk, but a higher beta does not guarantee a higher return. A one-year estimate may better reflect recent changes in a company, though it can also be less stable than a longer estimate. If our manual calculation is unavailable, the code falls back to Yahoo Finance’s beta and then to 1.0 if no value is available.

We select stocks using their scores, then assign a target 40% of the portfolio to Technology and Communication Services and 60% to stocks in other sectors. Within each group, higher-scoring stocks receive more weight. We cap each holding at 15% to limit the impact of any one company. This combines a growth focus with some diversification across holdings.

### Explanation 2 — Evidence and equal-weight comparison

We compare our score-weighted portfolio with an equal-weight portfolio holding the same stocks over the same dates. The notebook reports historical price return and daily CAD volatility for both portfolios. Since both contain the same stocks, this comparison helps us assess how our weighting method affected the results.

We use return to see how the portfolio’s value changed, and volatility to see how much its daily returns moved around. The comparison table gives us the statistics to evaluate the trade-off: whether our weighting produced a higher or lower historical return, and whether it came with higher or lower day-to-day volatility. This comparison evaluates our weights; it does not show whether our stock selection outperformed the overall market.

Our score and comparison use the same previous-year period, so we treat the results as a historical illustration rather than an independent test or a forecast.

### Explanation 3 — What could go wrong?

Our strategy relies on past returns and beta, but both can change. A one-year beta may reflect recent market conditions better than a five-year beta, especially when a company’s business has changed. However, a shorter period can also make beta less stable and more affected by unusual market events.

A high-beta portfolio may lose more than the market during a downturn. The 15% limit reduces the effect of a large loss in one holding, but it cannot prevent several holdings from falling at the same time. The sector allocation can also leave us exposed if growth-focused industries or other sectors perform poorly. Our score does not assess company valuation, debt, or earnings quality, and exchange-rate changes can affect the CAD value of US holdings.